# Realized Price Stop Test

**Hypothesis:** Adding a Realized Price floor stop protects against deep bear markets.

**The Concept:**
- Realized Price = Realized Cap / Supply = Average cost basis of all BTC
- Price < Realized Price = Network is underwater on average
- This only happens in deep bear markets (2018-2019, 2022)
- Exit when price crosses below to preserve gains

**Testing on:**
- STRAT-002 (Long-term): Simple 30% trail + Realized Price stop
- STRAT-003 (Short-term): LTH-SOPR exit + Realized Price stop

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from numba import njit
import warnings
warnings.filterwarnings('ignore')

print("Realized Price Stop Test 📊")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
sopr_lth = pd.read_parquet(DATA_DIR / "sopr_lth.parquet").rename(columns={"value": "sopr_lth"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
mvrv_z = pd.read_parquet(DATA_DIR / "mvrv_z.parquet").rename(columns={"value": "mvrv_z"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")
realized_price = pd.read_parquet(DATA_DIR / "realized_price.parquet").rename(columns={"value": "realized_price"}).set_index("time")

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(sopr_lth, how='inner')
df = df.join(mvrv, how='inner').join(mvrv_z, how='inner').join(realized_loss, how='inner').join(realized_price, how='inner')
df = df.sort_index()
df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()
df = df[df.index >= '2019-01-01'].dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")
print(f"\nRealized Price stats:")
print(f"  Current: ${df['realized_price'].iloc[-1]:,.0f}")
print(f"  Current Price: ${df['price'].iloc[-1]:,.0f}")
print(f"  Price/RP ratio: {df['price'].iloc[-1] / df['realized_price'].iloc[-1]:.2f}x")

---
## 1. Visualize Price vs Realized Price

In [ ]:
# Calculate when price was below realized price
df['below_rp'] = df['price'] < df['realized_price']
df['price_rp_ratio'] = df['price'] / df['realized_price']

print("PRICE vs REALIZED PRICE ANALYSIS")
print("="*60)
print(f"Days price < realized price: {df['below_rp'].sum()} ({df['below_rp'].mean()*100:.1f}%)")
print(f"\nPeriods below realized price:")

# Find contiguous periods below RP
below_periods = []
in_period = False
start = None

for i, (date, row) in enumerate(df.iterrows()):
    if row['below_rp'] and not in_period:
        in_period = True
        start = date
    elif not row['below_rp'] and in_period:
        in_period = False
        below_periods.append((start, date, (date - start).days))

if in_period:
    below_periods.append((start, df.index[-1], (df.index[-1] - start).days))

for start, end, days in below_periods:
    min_price = df.loc[start:end, 'price'].min()
    min_rp = df.loc[start:end, 'realized_price'].mean()
    print(f"  {start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}: {days} days (min ${min_price:,.0f}, RP ~${min_rp:,.0f})")

In [ ]:
# Visualize
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.5, 0.25, 0.25],
    subplot_titles=('BTC Price vs Realized Price', 'Price/RP Ratio', 'MVRV')
)

# Price and Realized Price
fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price', line=dict(color='orange')), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['realized_price'], name='Realized Price', line=dict(color='blue', dash='dash')), row=1, col=1)

# Highlight periods below RP
for start, end, days in below_periods:
    fig.add_vrect(x0=start, x1=end, fillcolor='red', opacity=0.2, line_width=0, row=1, col=1)

# Price/RP ratio
fig.add_trace(go.Scatter(x=df.index, y=df['price_rp_ratio'], name='Price/RP', line=dict(color='green')), row=2, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='red', row=2, col=1)

# MVRV
fig.add_trace(go.Scatter(x=df.index, y=df['mvrv'], name='MVRV', line=dict(color='purple')), row=3, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='red', row=3, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=900, title='Price vs Realized Price (Red zones = Price below RP)', showlegend=True)
fig.show()

---
## 2. Forward Returns When Below Realized Price

In [ ]:
# Forward returns
df['fwd_30d'] = df['price'].shift(-30) / df['price'] - 1
df['fwd_90d'] = df['price'].shift(-90) / df['price'] - 1

print("FORWARD RETURNS ANALYSIS")
print("="*70)
print(f"\n{'Condition':<35} {'Avg 30d':>12} {'Avg 90d':>12} {'Days':>8}")
print("-"*70)

conditions = [
    ('All days', df['price'] > 0),
    ('Price > Realized Price', ~df['below_rp']),
    ('Price < Realized Price', df['below_rp']),
    ('Price/RP < 0.9', df['price_rp_ratio'] < 0.9),
    ('Price/RP < 0.8', df['price_rp_ratio'] < 0.8),
    ('MVRV < 1', df['mvrv'] < 1),
]

for name, cond in conditions:
    subset = df[cond]
    if len(subset) > 5:
        avg_30 = subset['fwd_30d'].mean() * 100
        avg_90 = subset['fwd_90d'].mean() * 100
        print(f"{name:<35} {avg_30:>+11.1f}% {avg_90:>+11.1f}% {len(subset):>8}")

In [ ]:
print("\n💡 KEY INSIGHT:")
print("="*60)
above_rp = df[~df['below_rp']]
below_rp = df[df['below_rp']]

if len(below_rp) > 0:
    print(f"\nWhen Price > Realized Price:")
    print(f"  Avg 90d return: {above_rp['fwd_90d'].mean()*100:+.1f}%")
    print(f"\nWhen Price < Realized Price:")
    print(f"  Avg 90d return: {below_rp['fwd_90d'].mean()*100:+.1f}%")
    print(f"\nThis means price below RP is actually a GOOD time to hold!")
    print(f"(Buying opportunity, not exit signal)")
else:
    print("No periods with price below realized price in this timeframe.")

---
## 3. Backtest Exit Strategies

In [ ]:
@njit
def exit_simple_trail(price_arr, rp_arr, lth_arr, mvrv_arr, entry_idx, trail_pct=0.30):
    """Baseline: Simple trailing stop"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        if price > peak:
            peak = price
        if price <= peak * (1 - trail_pct):
            return j, price, 'trail'
    return len(price_arr) - 1, price_arr[-1], 'hold'


@njit
def exit_trail_with_rp_stop(price_arr, rp_arr, lth_arr, mvrv_arr, entry_idx, trail_pct=0.30):
    """Trailing stop + Realized Price floor"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        rp = rp_arr[j]
        
        if price > peak:
            peak = price
        
        # Exit if price drops below realized price (bear market floor)
        if price < rp:
            return j, price, 'rp_stop'
        
        # Normal trailing stop
        if price <= peak * (1 - trail_pct):
            return j, price, 'trail'
    
    return len(price_arr) - 1, price_arr[-1], 'hold'


@njit
def exit_lth_trigger(price_arr, rp_arr, lth_arr, mvrv_arr, entry_idx,
                     mvrv_context=2.5, lth_exit=1.5,
                     trail_before=0.30, trail_after=0.15):
    """MVRV + LTH-SOPR triggered trail (STRAT-003 v2)"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    triggered = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        lth = lth_arr[j]
        mvrv = mvrv_arr[j]
        
        if price > peak:
            peak = price
        
        if not triggered:
            if mvrv > mvrv_context and lth > lth_exit:
                triggered = True
        
        trail = trail_after if triggered else trail_before
        
        if price <= peak * (1 - trail):
            reason = 'lth_trail' if triggered else 'trail'
            return j, price, reason
    
    return len(price_arr) - 1, price_arr[-1], 'hold'


@njit
def exit_lth_trigger_with_rp(price_arr, rp_arr, lth_arr, mvrv_arr, entry_idx,
                              mvrv_context=2.5, lth_exit=1.5,
                              trail_before=0.30, trail_after=0.15):
    """MVRV + LTH-SOPR triggered trail + Realized Price floor"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    triggered = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        rp = rp_arr[j]
        lth = lth_arr[j]
        mvrv = mvrv_arr[j]
        
        if price > peak:
            peak = price
        
        # Exit if price drops below realized price (bear market floor)
        if price < rp:
            return j, price, 'rp_stop'
        
        if not triggered:
            if mvrv > mvrv_context and lth > lth_exit:
                triggered = True
        
        trail = trail_after if triggered else trail_before
        
        if price <= peak * (1 - trail):
            reason = 'lth_trail' if triggered else 'trail'
            return j, price, reason
    
    return len(price_arr) - 1, price_arr[-1], 'hold'


@njit
def exit_trail_with_rp_buffer(price_arr, rp_arr, lth_arr, mvrv_arr, entry_idx, 
                               trail_pct=0.30, rp_buffer=0.95):
    """Trailing stop + Realized Price with buffer (exit at 95% of RP)"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        rp = rp_arr[j]
        
        if price > peak:
            peak = price
        
        # Exit if price drops to X% of realized price
        if price < rp * rp_buffer:
            return j, price, 'rp_buffer'
        
        if price <= peak * (1 - trail_pct):
            return j, price, 'trail'
    
    return len(price_arr) - 1, price_arr[-1], 'hold'

In [ ]:
def run_backtest(df, entries, exit_func, initial_capital=100000, **kwargs):
    price_arr = df['price'].values
    rp_arr = df['realized_price'].values
    lth_arr = df['sopr_lth'].values
    mvrv_arr = df['mvrv'].values
    dates = df.index
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        exit_idx, exit_price, exit_reason = exit_func(price_arr, rp_arr, lth_arr, mvrv_arr, entry_idx, **kwargs)
        
        entry_price = price_arr[entry_idx]
        net_return = (exit_price / entry_price) - 1 - 0.002
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'exit_price': exit_price,
            'exit_reason': exit_reason,
            'exit_rp': rp_arr[exit_idx],
            'exit_mvrv': mvrv_arr[exit_idx],
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    return trades_df


def calc_metrics(trades, initial_capital=100000):
    if len(trades) == 0:
        return None
    final = trades['equity'].iloc[-1]
    total_ret = (final / initial_capital) - 1
    years = (trades['exit_date'].iloc[-1] - trades['entry_date'].iloc[0]).days / 365.25
    win_rate = (trades['net_return'] > 0).mean()
    returns = trades['net_return'].values
    sharpe = (returns.mean() / returns.std()) * np.sqrt(len(trades)/years) if returns.std() > 0 and years > 0 else 0
    
    equity = [initial_capital] + list(trades['equity'])
    peak, max_dd = equity[0], 0
    for eq in equity:
        if eq > peak: peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd: max_dd = dd
    
    return {
        'total_return': total_ret,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'win_rate': win_rate,
        'n_trades': len(trades),
        'avg_hold': trades['days_held'].mean()
    }

---
## 4. STRAT-002 (Long-term) with Realized Price Stop

In [ ]:
# STRAT-002 Entry
entry_cond_002 = (df['sopr'] < 1) & (df['sopr_sth'] < 1) & (df['rl_zscore'] > 0.5)
entries_002 = entry_cond_002 & ~entry_cond_002.shift(1).fillna(False)

print("STRAT-002 (LONG-TERM) + REALIZED PRICE STOP")
print("Entry: SOPR < 1 AND STH-SOPR < 1 AND RL Z > 0.5")
print("="*100)

strategies_002 = [
    ('Simple 30% Trail (baseline)', exit_simple_trail, {'trail_pct': 0.30}),
    ('30% Trail + RP Stop', exit_trail_with_rp_stop, {'trail_pct': 0.30}),
    ('30% Trail + RP*0.95 Stop', exit_trail_with_rp_buffer, {'trail_pct': 0.30, 'rp_buffer': 0.95}),
    ('30% Trail + RP*0.90 Stop', exit_trail_with_rp_buffer, {'trail_pct': 0.30, 'rp_buffer': 0.90}),
    ('25% Trail + RP Stop', exit_trail_with_rp_stop, {'trail_pct': 0.25}),
    ('20% Trail + RP Stop', exit_trail_with_rp_stop, {'trail_pct': 0.20}),
]

print(f"{'Strategy':<35} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'Win%':>7} {'Trades':>7} {'AvgHold':>8}")
print("-"*95)

results_002 = []
for name, func, kwargs in strategies_002:
    trades = run_backtest(df, entries_002, func, **kwargs)
    m = calc_metrics(trades)
    if m:
        print(f"{name:<35} {m['total_return']*100:>+9.0f}% {m['sharpe']:>8.2f} {m['max_dd']*100:>7.0f}% {m['win_rate']*100:>6.0f}% {m['n_trades']:>7} {m['avg_hold']:>7.0f}d")
        results_002.append({'name': name, 'metrics': m, 'trades': trades})

In [ ]:
# Show trades for RP stop version
rp_result = [r for r in results_002 if 'RP Stop' in r['name'] and '0.9' not in r['name'] and '0.95' not in r['name']][0]
print(f"\nTRADE LOG: {rp_result['name']}")
print("="*100)

t = rp_result['trades']
print(f"\n{'Entry':<12} {'Exit':<12} {'Days':>6} {'Return':>10} {'Reason':<12} {'Exit Price':>12} {'Exit RP':>12}")
print("-"*90)

for _, row in t.iterrows():
    print(f"{row['entry_date'].strftime('%Y-%m-%d'):<12} {row['exit_date'].strftime('%Y-%m-%d'):<12} {row['days_held']:>6} {row['net_return']*100:>+9.0f}% {row['exit_reason']:<12} ${row['exit_price']:>11,.0f} ${row['exit_rp']:>11,.0f}")

---
## 5. STRAT-003 (Short-term) with Realized Price Stop

In [ ]:
# STRAT-003 Entry
entry_cond_003 = df['sopr_sth'] < 1
entries_003 = entry_cond_003 & ~entry_cond_003.shift(1).fillna(False)

print("\nSTRAT-003 (SHORT-TERM) + REALIZED PRICE STOP")
print("Entry: STH-SOPR < 1")
print("="*100)

strategies_003 = [
    ('Simple 30% Trail', exit_simple_trail, {'trail_pct': 0.30}),
    ('LTH trigger (v2 baseline)', exit_lth_trigger,
     {'mvrv_context': 2.5, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('LTH trigger + RP Stop', exit_lth_trigger_with_rp,
     {'mvrv_context': 2.5, 'lth_exit': 1.5, 'trail_before': 0.30, 'trail_after': 0.15}),
    ('30% Trail + RP Stop', exit_trail_with_rp_stop, {'trail_pct': 0.30}),
    ('30% Trail + RP*0.95', exit_trail_with_rp_buffer, {'trail_pct': 0.30, 'rp_buffer': 0.95}),
]

print(f"{'Strategy':<35} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'Win%':>7} {'Trades':>7} {'AvgHold':>8}")
print("-"*95)

results_003 = []
for name, func, kwargs in strategies_003:
    trades = run_backtest(df, entries_003, func, **kwargs)
    m = calc_metrics(trades)
    if m:
        print(f"{name:<35} {m['total_return']*100:>+9.0f}% {m['sharpe']:>8.2f} {m['max_dd']*100:>7.0f}% {m['win_rate']*100:>6.0f}% {m['n_trades']:>7} {m['avg_hold']:>7.0f}d")
        results_003.append({'name': name, 'metrics': m, 'trades': trades})

In [ ]:
# Count RP stop triggers
print("\n" + "="*60)
print("RP STOP TRIGGER ANALYSIS")
print("="*60)

for result in results_002 + results_003:
    if 'RP' in result['name']:
        t = result['trades']
        rp_exits = (t['exit_reason'] == 'rp_stop').sum() + (t['exit_reason'] == 'rp_buffer').sum()
        total = len(t)
        print(f"{result['name']:<40}: {rp_exits}/{total} trades exited via RP stop")

---
## 6. Summary

In [ ]:
print("\n" + "="*70)
print("REALIZED PRICE STOP SUMMARY")
print("="*70)

# Find best for each strategy
best_002_base = [r for r in results_002 if 'baseline' in r['name']][0]
best_002_rp = max([r for r in results_002 if 'RP' in r['name']], key=lambda x: x['metrics']['total_return'])

best_003_base = [r for r in results_003 if 'v2 baseline' in r['name']][0]
best_003_rp = max([r for r in results_003 if 'RP' in r['name']], key=lambda x: x['metrics']['total_return'])

print(f"""
📊 STRAT-002 (Long-term):
   Baseline (30% trail): {best_002_base['metrics']['total_return']*100:+,.0f}%
   Best with RP Stop: {best_002_rp['metrics']['total_return']*100:+,.0f}% ({best_002_rp['name']})
   Winner: {'RP Stop ✅' if best_002_rp['metrics']['total_return'] > best_002_base['metrics']['total_return'] else 'Baseline ✅'}

📊 STRAT-003 (Short-term):
   Baseline (LTH trigger): {best_003_base['metrics']['total_return']*100:+,.0f}%
   Best with RP Stop: {best_003_rp['metrics']['total_return']*100:+,.0f}% ({best_003_rp['name']})
   Winner: {'RP Stop ✅' if best_003_rp['metrics']['total_return'] > best_003_base['metrics']['total_return'] else 'Baseline ✅'}

💡 KEY INSIGHT:
   • Price < Realized Price is RARE (only deep bear markets)
   • Forward returns when below RP are actually POSITIVE
   • RP marks BOTTOMS, not tops!
   • Using RP as exit would sell at worst possible time
""")